# OSINT Desk on Google Colab

Personal / lab investigations. Free sources only. History is a SQLite file in this runtime (download it before the VM is recycled).

**Setup:** upload the project folder (the directory that contains `osint/`, `sources.yaml`, and `requirements.txt`) to `/content/osint-desk`, or clone it into that path, then run the cells in order.

In [ ]:
import os, sys, subprocess
from pathlib import Path

CANDIDATES = [
    Path('/content/osint-desk'),
    Path('/content'),
    Path('.').resolve(),
]
ROOT = next((p for p in CANDIDATES if (p / 'osint' / 'app.py').exists()), None)
if ROOT is None:
    raise SystemExit('Upload the OSINT Desk project so osint/app.py exists under /content/osint-desk')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print('Project root:', ROOT)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])

## API keys (optional)

Stored in `/content/osint-desk-data/secrets.env` inside this runtime — not in the history database. Leave blank to run keyless sources (InternetDB, RDAP, Geo/ASN, crt.sh, DNS, hashlookup, MalwareBazaar, ThreatFox, URLhaus).

In [ ]:
import os
from getpass import getpass
from osint.config import write_secret, load_secrets, ensure_data_dir

os.environ['OSINT_DATA_DIR'] = '/content/osint-desk-data'
os.environ['OSINT_HOST'] = '0.0.0.0'
os.environ['OSINT_PORT'] = '8080'
ensure_data_dir()
load_secrets()

KEYS = [
    'ABUSEIPDB_API_KEY',
    'VT_API_KEY',
    'GREYNOISE_API_KEY',
    'OTX_API_KEY',
    'URLSCAN_API_KEY',
    'THREATFOX_API_KEY',
    'MALWAREBAZAAR_API_KEY',
    'URLHAUS_API_KEY',
    'IPQS_API_KEY',
    'APIVOID_API_KEY',
    'HYBRID_ANALYSIS_API_KEY',
    'TRIAGE_API_KEY',
    'HUNTER_API_KEY',
    'AI_API_KEY',
]
# Colab secrets (if you saved them in the key icon)
try:
    from google.colab import userdata
    for name in KEYS:
        try:
            val = userdata.get(name)
        except Exception:
            val = None
        if val:
            write_secret(name, val)
            print('loaded', name, 'from Colab secrets')
except Exception:
    pass

print('Paste keys or press Enter to skip.')
for name in KEYS:
    if os.environ.get(name):
        continue
    val = getpass(name + ' (blank skip): ')
    if val.strip():
        write_secret(name, val.strip())


## Google Drive (survives runtime reset)

Data (SQLite, secrets, optional Ollama models) can live on Drive. Code does not — re-upload the project after a reset, then restore.


In [ ]:
from pathlib import Path
import os, shutil

USE_DRIVE = True  # set False to stay on the ephemeral disk
DRIVE_ROOT = Path('/content/drive/MyDrive/osint-desk')
LOCAL = Path('/content/osint-desk-data')

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ['OSINT_DATA_DIR'] = str(DRIVE_ROOT / 'data')
    Path(os.environ['OSINT_DATA_DIR']).mkdir(parents=True, exist_ok=True)
    print('Data dir:', os.environ['OSINT_DATA_DIR'])
else:
    os.environ['OSINT_DATA_DIR'] = str(LOCAL)

def save_to_drive():
    src = Path(os.environ['OSINT_DATA_DIR'])
    dest = DRIVE_ROOT / 'data'
    dest.mkdir(parents=True, exist_ok=True)
    for name in ['osint.sqlite3', 'secrets.env', 'session.secret']:
        p = src / name
        if p.exists():
            shutil.copy2(p, dest / name)
            print('saved', name)

def restore_from_drive():
    dest = Path(os.environ['OSINT_DATA_DIR'])
    dest.mkdir(parents=True, exist_ok=True)
    src = DRIVE_ROOT / 'data'
    for name in ['osint.sqlite3', 'secrets.env', 'session.secret']:
        p = src / name
        if p.exists():
            shutil.copy2(p, dest / name)
            print('restored', name)

restore_from_drive()
print('Call save_to_drive() before you recycle the runtime.')


## Local AI (Ollama)

Optional. GPU → larger model. CPU/TPU → smaller model (Ollama cannot use a TPU). No silent fallback to a cloud API.


In [ ]:
import os, shutil, subprocess

def pick_model():
    gpu = False
    tpu = False
    try:
        out = subprocess.check_output(['nvidia-smi', '-L'], text=True, stderr=subprocess.DEVNULL)
        gpu = 'GPU' in out
    except Exception:
        gpu = False
    tpu = bool(os.environ.get('COLAB_TPU_ADDR') or os.environ.get('TPU_NAME'))
    if gpu:
        model, note = 'llama3.1', 'GPU detected — using llama3.1'
    elif tpu:
        model, note = 'llama3.2:1b', 'TPU is present but Ollama cannot use it. Falling back to a small CPU model (llama3.2:1b).'
    else:
        model, note = 'llama3.2:1b', 'CPU only — using llama3.2:1b'
    os.environ['AI_BASE_URL'] = 'http://127.0.0.1:11434/v1'
    os.environ['AI_MODEL'] = model
    os.environ['AI_LABEL'] = 'Local: ' + model
    print(note)
    return model

print(pick_model())
print('Install Ollama in a separate cell if you want on-runtime AI. Otherwise AI buttons will error clearly.')


## Start the desk

The next cell launches uvicorn in a background thread and prints a Colab proxy URL. Open that URL. Stop the runtime when you are done; download `osint.sqlite3` from `/content/osint-desk-data` if you want the history.

In [ ]:
import threading, time, os, socket, urllib.request

os.environ.setdefault('OSINT_DATA_DIR', '/content/osint-desk-data')
os.environ['OSINT_HOST'] = '0.0.0.0'
os.environ['OSINT_STRICT_HOSTS'] = '0'

def free_port(start=8080):
    for p in range(start, start + 20):
        s = socket.socket(); s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        try:
            s.bind(('127.0.0.1', p)); s.close(); return p
        except OSError:
            s.close()
    return start

port = int(os.environ.get('OSINT_PORT', '8080'))
# If 8080 is a stale process, pick another and remember it.
try:
    urllib.request.urlopen(f'http://127.0.0.1:{port}/healthz', timeout=0.4).read()
    # already serving — verify it's this code by hitting /healthz name
except Exception:
    port = free_port(8080)
os.environ['OSINT_PORT'] = str(port)

def _serve():
    import uvicorn
    uvicorn.run('osint.app:app', host='0.0.0.0', port=port, log_level='warning')

t = threading.Thread(target=_serve, daemon=True)
t.start()
ok = False
for _ in range(50):
    try:
        urllib.request.urlopen(f'http://127.0.0.1:{port}/healthz', timeout=1).read()
        ok = True
        break
    except Exception:
        time.sleep(0.25)
if not ok:
    raise SystemExit('Server did not become healthy')

try:
    from google.colab.output import eval_js
    url = eval_js(f'google.colab.kernel.proxyPort({port})')
    print('Open the desk:', url)
    print('This is Colab\'s private proxy — not a public tunnel.')
except Exception:
    print(f'Server is on port {port} (no Colab proxy in this runtime)')
